In [ ]:
import pandas as pd
import csv

In [ ]:
train = pd.read_csv(r"C:\Users\SCT\Downloads\tr_train.csv")
test = pd.read_csv(r"C:\Users\SCT\Downloads\tr_test.csv")

#VERİ ÇEKME


In [ ]:
train.shape, test.shape
#SHAPELERİ KONTROL

In [ ]:
train["ikili"].value_counts()
#DENGE KOTROLÜ

In [ ]:
train["belirsiz"].value_counts(dropna=False), test["belirsiz"].value_counts(dropna=False)


In [ ]:
#BELİRSİZLERDEN KURTULUYORUM
train = train[train["belirsiz"] != "e"]
X_train = train["mesaj"]
y_train = train["ikili"]
X_test = test["mesaj"]
y_test = test["ikili"]
print(X_train.shape, X_test.shape)
print(y_train.value_counts())

In [ ]:
#TF-IDF

from sklearn.feature_extraction.text import TfidfVectorizer
vec = TfidfVectorizer()
X_train_vec = vec.fit_transform(X_train)
X_test_vec = vec.transform(X_test)
print(X_train_vec.shape, X_test_vec.shape)
print(len(vec.vocabulary_))

In [ ]:
#Naif Bayes :D

from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

nb = MultinomialNB()
nb.fit(X_train_vec, y_train)
y_pred_nb = nb.predict(X_test_vec)
print(classification_report(y_test, y_pred_nb, digits = 3))

In [ ]:
hatalar = test[y_test != y_pred_nb]
hatalar[["id", "mesaj", "ikili", "kaynak_etiket"]]

In [ ]:
#HATALARI ANALİZ ETMECE

kelimeler = vec.get_feature_names_out()
fark = nb.feature_log_prob_[1] - nb.feature_log_prob_[0]

def neden(i):
    idx = X_test_vec[i].nonzero()[1]
    return sorted(zip(kelimeler[idx], fark[idx]), key=lambda t: -t[1])

print(nb.classes_)
print(neden(7))
print(neden(27))

In [ ]:
#SVM, naif fena değil ama veri seti küçük olduğundan dolayı SVM ile denemece.
from sklearn.svm import LinearSVC
svm = LinearSVC(random_state=55)
svm.fit(X_train_vec, y_train)
y_pred_svm = svm.predict(X_test_vec)
print(classification_report(y_test, y_pred_svm, digits = 3))
print(test[y_test != y_pred_svm][["id", "mesaj", "ikili",]])

In [ ]:
vec_c = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5))
X_train_vec_c = vec_c.fit_transform(X_train)
X_test_vec_c = vec_c.transform(X_test)
print(X_train_vec_c.shape)

for ad, model in ["NB", MultinomialNB()], ["SVM", LinearSVC(random_state=55)]:
    model.fit(X_train_vec_c, y_train)
    tahmin = model.predict(X_test_vec_c)
    print(ad)
    print(classification_report(y_test, tahmin, digits = 3))
    print(test[y_test != tahmin]["id"].tolist())

In [ ]:
train["fold"].value_counts().sort_index()

In [ ]:
#CROSS VALIDATION İÇİN PIPELINE 

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import PredefinedSplit, cross_val_score
cv = PredefinedSplit(train["fold"])

adaylar = {"NB kelime": make_pipeline(TfidfVectorizer(), MultinomialNB()),
           "SVM kelime":make_pipeline(TfidfVectorizer(), LinearSVC(random_state=55)),
           "NB harf":   make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5)), MultinomialNB()),
           "SVM harf" : make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5)), LinearSVC(random_state=55))}
for ad, model in adaylar.items():
    skor = cross_val_score(model,X_train, y_train, cv=cv, scoring="f1_macro")
    print(f"{ad}: {skor.mean():.3f} ± {skor.std():.3f} {skor.round(3)}")



In [ ]:
import joblib
from pathlib import Path

final = adaylar["SVM harf"]
final.fit(X_train, y_train)
print(classification_report(y_test, final.predict(X_test), digits = 3))

Path("../models").mkdir(exist_ok=True)
joblib.dump(final, "../models/svm_char_tr.joblib")

yuklenen = joblib.load("../models/svm_char_tr.joblib")
print(yuklenen.predict(["kanka aksam maca geliyor musun"
                        , "Tebrikler! 5000 TL kazandiniz, "
                        "hemen tiklayin"]))
